# Projet CNN — Classification d'images CIFAR-10
**IA, Deep Learning et Applications - E3FD**  
**Présentation : 13 avril 2026**

---

## Plan du projet
1. Définition du problème et du dataset
2. Préparation du dataset
3. Première architecture CNN
4. Réduction de l'overfitting / amélioration des performances
5. Discussion et comparaison des modèles

## 0. Imports et configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU disponible : {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 1. Définition du problème et du dataset

In [ ]:
# Chargement du dataset CIFAR-10
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

# Noms des classes
CLASS_NAMES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
NUM_CLASSES = 10

print("=== Description du dataset CIFAR-10 ===")
print(f"Ensemble d'entraînement : {X_train_raw.shape}  — {X_train_raw.shape[0]} images")
print(f"Ensemble de test        : {X_test_raw.shape}  — {X_test_raw.shape[0]} images")
print(f"Taille d'une image      : {X_train_raw.shape[1:]}  (H x W x C)")
print(f"Nombre de classes       : {NUM_CLASSES}")
print(f"Plage de valeurs pixels : [{X_train_raw.min()}, {X_train_raw.max()}]")

In [ ]:
# Visualisation — exemples du dataset
fig, axes = plt.subplots(4, 10, figsize=(20, 9))
fig.suptitle('Exemples du dataset CIFAR-10 (4 images par classe)', fontsize=14, fontweight='bold')

for col, class_name in enumerate(CLASS_NAMES):
    indices = np.where(y_train_raw.flatten() == col)[0][:4]
    for row, idx in enumerate(indices):
        axes[row, col].imshow(X_train_raw[idx])
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(class_name, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_dataset_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : fig_dataset_examples.png")

In [ ]:
# Distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des classes', fontsize=13, fontweight='bold')

for ax, labels, title in zip(axes,
                              [y_train_raw, y_test_raw],
                              ['Entraînement (50 000 images)', 'Test (10 000 images)']):
    counts = [np.sum(labels == i) for i in range(NUM_CLASSES)]
    bars = ax.bar(CLASS_NAMES, counts, color=plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES)))
    ax.set_title(title)
    ax.set_ylabel('Nombre d\'images')
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ Dataset équilibré : 5 000 images par classe en entraînement, 1 000 en test.")

---
## 2. Préparation du dataset

In [ ]:
# --- 2.1 Normalisation des images (float32, [0, 1]) ---
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32') / 255.0

print(f"Après normalisation — plage : [{X_train.min():.1f}, {X_train.max():.1f}]")

# --- 2.2 Partition train / validation (80/20) ---
VAL_SPLIT = 0.2
n_val = int(len(X_train) * VAL_SPLIT)

X_val   = X_train[-n_val:]
y_val   = y_train_raw[-n_val:]
X_train = X_train[:-n_val]
y_train = y_train_raw[:-n_val]

print(f"\nPartition des ensembles :")
print(f"  Entraînement : {X_train.shape[0]:>6} images")
print(f"  Validation   : {X_val.shape[0]:>6} images")
print(f"  Test         : {X_test.shape[0]:>6} images")

# --- 2.3 Encodage one-hot des labels ---
y_train_ohe = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_ohe   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_ohe  = keras.utils.to_categorical(y_test_raw, NUM_CLASSES)

print(f"\nShape des labels encodés (one-hot) :")
print(f"  y_train_ohe : {y_train_ohe.shape}")
print(f"  y_val_ohe   : {y_val_ohe.shape}")
print(f"  y_test_ohe  : {y_test_ohe.shape}")

---
## 3. Implémentation de la première architecture CNN

In [ ]:
# ===================================================
# MODÈLE 1 — Architecture CNN de base
# ===================================================
def build_model_base():
    model = keras.Sequential([
        # Bloc convolutif 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      input_shape=(32, 32, 3), name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),

        # Bloc convolutif 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),

        # Bloc convolutif 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3'),
        layers.MaxPooling2D((2, 2), name='pool3'),

        # Classifieur
        layers.Flatten(name='flatten'),
        layers.Dense(256, activation='relu', name='dense1'),
        layers.Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='CNN_Base')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model1 = build_model_base()
model1.summary()

In [ ]:
# Entraînement du modèle 1
EPOCHS = 30
BATCH_SIZE = 64

print("Entraînement du Modèle 1 (CNN de base)...")
t_start = time.time()

history1 = model1.fit(
    X_train, y_train_ohe,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_ohe),
    verbose=1
)

t1_duration = time.time() - t_start
print(f"\nDurée d'entraînement : {t1_duration:.1f} s")

# Évaluation sur le test
test_loss1, test_acc1 = model1.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Test accuracy : {test_acc1*100:.2f}%  |  Test loss : {test_loss1:.4f}")

# Sauvegarde
model1.save('model1_base.keras')
with open('history1_base.pickle', 'wb') as f:
    pickle.dump(history1.history, f)
print("Modèle sauvegardé : model1_base.keras")
print("Historique sauvegardé : history1_base.pickle")

In [ ]:
def plot_history(history_dict, title, filename):
    """Trace les courbes loss et accuracy train/val."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Loss
    axes[0].plot(history_dict['loss'],     label='Train loss',      color='steelblue')
    axes[0].plot(history_dict['val_loss'], label='Validation loss', color='tomato', linestyle='--')
    axes[0].set_title('Fonction de perte (Loss)')
    axes[0].set_xlabel('Époque')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(history_dict['accuracy'],     label='Train accuracy',      color='steelblue')
    axes[1].plot(history_dict['val_accuracy'], label='Validation accuracy', color='tomato', linestyle='--')
    axes[1].set_title('Précision (Accuracy)')
    axes[1].set_xlabel('Époque')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure sauvegardée : {filename}")

# Affichage des courbes du modèle 1
plot_history(history1.history,
             'Modèle 1 — CNN de base (30 époques)',
             'fig_model1_curves.png')

# Diagnostic de l'overfitting
final_train_acc = history1.history['accuracy'][-1]
final_val_acc   = history1.history['val_accuracy'][-1]
gap = final_train_acc - final_val_acc
print(f"\nDiagnostic overfitting :")
print(f"  Accuracy entraînement  : {final_train_acc*100:.2f}%")
print(f"  Accuracy validation    : {final_val_acc*100:.2f}%")
print(f"  Écart (gap)            : {gap*100:.2f}%")
if gap > 0.10:
    print("  → Overfitting détecté (gap > 10%). Des régularisations sont nécessaires.")
else:
    print("  → Pas d'overfitting significatif détecté.")

---
## 4. Réduction de l'overfitting et amélioration des performances

Nous allons implémenter et comparer **3 stratégies** :

| Modèle | Techniques |  
|--------|------------|  
| Modèle 2 | Dropout + Early Stopping |
| Modèle 3 | Dropout + L2 Regularization + Early Stopping |
| Modèle 4 | Data Augmentation + architecture plus profonde + Early Stopping |

In [ ]:
# ===================================================
# MODÈLE 2 — Dropout + Early Stopping
# ===================================================
def build_model_dropout():
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Dropout')

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model2 = build_model_dropout()

es2 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

print("Entraînement du Modèle 2 (Dropout + Early Stopping)...")
t_start = time.time()
history2 = model2.fit(
    X_train, y_train_ohe,
    epochs=50,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_ohe),
    callbacks=[es2],
    verbose=1
)
t2_duration = time.time() - t_start

test_loss2, test_acc2 = model2.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Durée : {t2_duration:.1f} s | Test accuracy : {test_acc2*100:.2f}%")

model2.save('model2_dropout.keras')
with open('history2_dropout.pickle', 'wb') as f:
    pickle.dump(history2.history, f)
print("Sauvegarde : model2_dropout.keras + history2_dropout.pickle")

plot_history(history2.history, 'Modèle 2 — Dropout + Early Stopping', 'fig_model2_curves.png')

In [ ]:
# ===================================================
# MODÈLE 3 — Dropout + Régularisation L2 + Early Stopping
# ===================================================
def build_model_l2():
    l2 = regularizers.L2(1e-4)

    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=l2, input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=l2),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=l2),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu', kernel_regularizer=l2),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_L2')

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model3 = build_model_l2()
es3 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

print("Entraînement du Modèle 3 (Dropout + L2 + Early Stopping)...")
t_start = time.time()
history3 = model3.fit(
    X_train, y_train_ohe,
    epochs=50,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val_ohe),
    callbacks=[es3],
    verbose=1
)
t3_duration = time.time() - t_start

test_loss3, test_acc3 = model3.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Durée : {t3_duration:.1f} s | Test accuracy : {test_acc3*100:.2f}%")

model3.save('model3_l2.keras')
with open('history3_l2.pickle', 'wb') as f:
    pickle.dump(history3.history, f)
print("Sauvegarde : model3_l2.keras + history3_l2.pickle")

plot_history(history3.history, 'Modèle 3 — Dropout + L2 + Early Stopping', 'fig_model3_curves.png')

In [ ]:
# ===================================================
# MODÈLE 4 — Data Augmentation + Architecture plus profonde
# ===================================================

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train)

def build_model_deep():
    model = keras.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        # Bloc 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Bloc 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),

        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Deep_Augmented')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model4 = build_model_deep()
model4.summary()

es4 = EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)

print("Entraînement du Modèle 4 (Data Augmentation + Architecture profonde)...")
t_start = time.time()
history4 = model4.fit(
    datagen.flow(X_train, y_train_ohe, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=60,
    validation_data=(X_val, y_val_ohe),
    callbacks=[es4],
    verbose=1
)
t4_duration = time.time() - t_start

test_loss4, test_acc4 = model4.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Durée : {t4_duration:.1f} s | Test accuracy : {test_acc4*100:.2f}%")

model4.save('model4_augmented.keras')
with open('history4_augmented.pickle', 'wb') as f:
    pickle.dump(history4.history, f)
print("Sauvegarde : model4_augmented.keras + history4_augmented.pickle")

plot_history(history4.history,
             'Modèle 4 — Data Augmentation + Architecture profonde',
             'fig_model4_curves.png')

---
## 5. Discussion — Comparaison des modèles

In [ ]:
# ===================================================
# Tableau comparatif des 4 modèles
# ===================================================
results = {
    'Modèle 1\n(Base)': {
        'test_acc': test_acc1,
        'train_acc': history1.history['accuracy'][-1],
        'val_acc': history1.history['val_accuracy'][-1],
        'duration': t1_duration,
        'epochs': len(history1.history['accuracy'])
    },
    'Modèle 2\n(Dropout)': {
        'test_acc': test_acc2,
        'train_acc': history2.history['accuracy'][-1],
        'val_acc': history2.history['val_accuracy'][-1],
        'duration': t2_duration,
        'epochs': len(history2.history['accuracy'])
    },
    'Modèle 3\n(Dropout+L2)': {
        'test_acc': test_acc3,
        'train_acc': history3.history['accuracy'][-1],
        'val_acc': history3.history['val_accuracy'][-1],
        'duration': t3_duration,
        'epochs': len(history3.history['accuracy'])
    },
    'Modèle 4\n(Augm.+Profond)': {
        'test_acc': test_acc4,
        'train_acc': history4.history['accuracy'][-1],
        'val_acc': history4.history['val_accuracy'][-1],
        'duration': t4_duration,
        'epochs': len(history4.history['accuracy'])
    }
}

print("=" * 80)
print(f"{'Modèle':<25} {'Test Acc':>10} {'Train Acc':>10} {'Val Acc':>10} {'Gap':>8} {'Durée (s)':>10} {'Époques':>8}")
print("-" * 80)
for name, r in results.items():
    gap = r['train_acc'] - r['val_acc']
    print(f"{name.replace(chr(10), ' '):<25} {r['test_acc']*100:>9.2f}% {r['train_acc']*100:>9.2f}% {r['val_acc']*100:>9.2f}% {gap*100:>7.2f}% {r['duration']:>10.1f} {r['epochs']:>8}")
print("=" * 80)

In [ ]:
# Graphique comparatif des performances
model_names = [k.replace('\n', '\n') for k in results.keys()]
test_accs  = [v['test_acc']*100  for v in results.values()]
train_accs = [v['train_acc']*100 for v in results.values()]
val_accs   = [v['val_acc']*100   for v in results.values()]
durations  = [v['duration']      for v in results.values()]

x = np.arange(len(model_names))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparaison des 4 modèles CNN', fontsize=14, fontweight='bold')

# Accuracy comparée
axes[0].bar(x - width, train_accs, width, label='Train', color='steelblue', alpha=0.85)
axes[0].bar(x,         val_accs,   width, label='Validation', color='tomato', alpha=0.85)
axes[0].bar(x + width, test_accs,  width, label='Test', color='seagreen', alpha=0.85)
axes[0].set_title('Précision (Accuracy) par ensemble')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(50, 100)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, (tr, va, te) in enumerate(zip(train_accs, val_accs, test_accs)):
    axes[0].text(i - width, tr + 0.3, f'{tr:.1f}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i,         va + 0.3, f'{va:.1f}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i + width, te + 0.3, f'{te:.1f}', ha='center', va='bottom', fontsize=8)

# Durée d'entraînement
colors = ['steelblue', 'tomato', 'darkorange', 'seagreen']
bars = axes[1].bar(model_names, durations, color=colors, alpha=0.85)
axes[1].set_title("Durée d'entraînement (secondes)")
axes[1].set_ylabel('Secondes')
axes[1].grid(axis='y', alpha=0.3)
for bar, d in zip(bars, durations):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{d:.0f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : fig_comparison.png")

In [ ]:
# Matrice de confusion — meilleur modèle
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Sélection du meilleur modèle (test accuracy max)
best_idx = np.argmax(test_accs)
best_models = [model1, model2, model3, model4]
best_model = best_models[best_idx]
best_name = list(results.keys())[best_idx].replace('\n', ' ')
print(f"Meilleur modèle : {best_name} ({test_accs[best_idx]:.2f}% sur le test)")

y_pred = np.argmax(best_model.predict(X_test), axis=1)
cm = confusion_matrix(y_test_raw.flatten(), y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title(f'Matrice de confusion — {best_name}', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : fig_confusion_matrix.png")

In [ ]:
# Exemples de prédictions (bonnes et mauvaises)
y_pred_all = np.argmax(best_model.predict(X_test), axis=1)
y_true_all = y_test_raw.flatten()

correct_idx = np.where(y_pred_all == y_true_all)[0][:8]
wrong_idx   = np.where(y_pred_all != y_true_all)[0][:8]

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle(f'Prédictions du {best_name}', fontsize=13, fontweight='bold')

for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(X_test_raw[idx])
    axes[0, i].axis('off')
    axes[0, i].set_title(f'✓ {CLASS_NAMES[y_true_all[idx]]}', color='green', fontsize=9)

for i, idx in enumerate(wrong_idx):
    axes[1, i].imshow(X_test_raw[idx])
    axes[1, i].axis('off')
    axes[1, i].set_title(f'✗ Vrai:{CLASS_NAMES[y_true_all[idx]]}\nPréd:{CLASS_NAMES[y_pred_all[idx]]}',
                         color='red', fontsize=8)

axes[0, 0].set_ylabel('Correctes', fontsize=11, rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('Erreurs', fontsize=11, rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.savefig('fig_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : fig_predictions.png")

In [ ]:
# Précision par classe
per_class_acc = []
for i in range(NUM_CLASSES):
    mask = y_true_all == i
    acc = np.mean(y_pred_all[mask] == y_true_all[mask])
    per_class_acc.append(acc * 100)
    print(f"  {CLASS_NAMES[i]:<12} : {acc*100:.1f}%")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(CLASS_NAMES, per_class_acc,
               color=[plt.cm.RdYlGn(v/100) for v in per_class_acc])
ax.set_xlabel('Accuracy (%)')
ax.set_title(f'Précision par classe — {best_name}', fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)
ax.axvline(np.mean(per_class_acc), color='navy', linestyle='--',
           label=f'Moyenne : {np.mean(per_class_acc):.1f}%')
ax.legend()
for bar, acc in zip(bars, per_class_acc):
    ax.text(acc + 0.5, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : fig_per_class_accuracy.png")

In [ ]:
# ===================================================
# SYNTHÈSE ET DISCUSSION
# ===================================================
print("=" * 70)
print("SYNTHÈSE DES RÉSULTATS")
print("=" * 70)

for name, r in results.items():
    gap = r['train_acc'] - r['val_acc']
    short_name = name.replace('\n', ' ')
    print(f"\n{short_name}")
    print(f"  Test accuracy    : {r['test_acc']*100:.2f}%")
    print(f"  Gap train-val    : {gap*100:.2f}%  "
          f"({'overfitting significatif' if gap > 0.10 else 'acceptable'})")
    print(f"  Durée            : {r['duration']:.1f}s sur {r['epochs']} époques")

print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)
print("""
• Modèle 1 (base) : Overfitting clair — le gap train/val dépasse souvent
  15-20%. Le modèle mémorise les données d'entraînement.

• Modèle 2 (Dropout) : Le Dropout réduit notablement l'overfitting.
  L'Early Stopping limite inutilement les époques superflues.

• Modèle 3 (Dropout + L2) : La régularisation L2 apporte une pénalisation
  supplémentaire sur les poids. Résultats similaires au modèle 2,
  parfois légèrement inférieurs si le λ est mal calibré.

• Modèle 4 (Augmentation + Profond) : La data augmentation artificielle
  augmente la diversité des données vues, réduisant fortement l'overfitting
  et améliorant la généralisation. C'est généralement le meilleur modèle.

LIMITES :
  - CIFAR-10 contient des images 32×32 basse résolution ;
    des architectures pré-entraînées (ResNet, EfficientNet) atteignent >95%.
  - Le temps d'entraînement du modèle 4 est nettement plus élevé.
  - L'hyperparameter tuning (learning rate, taux de dropout, λ L2)
    n'a pas été optimisé systématiquement.

PISTES D'AMÉLIORATION :
  - Transfer Learning (ResNet50, EfficientNetB0 pré-entraîné sur ImageNet)
  - Learning rate scheduling (cosine annealing, ReduceLROnPlateau)
  - Batch normalization généralisée
  - Augmentation plus agressive (Cutout, Mixup)
""")

In [ ]:
# Récapitulatif des fichiers sauvegardés
print("\n📁 Fichiers générés :")
saved_files = [
    ('model1_base.keras',         'Modèle 1 (base)'),
    ('history1_base.pickle',      'Historique Modèle 1'),
    ('model2_dropout.keras',      'Modèle 2 (Dropout)'),
    ('history2_dropout.pickle',   'Historique Modèle 2'),
    ('model3_l2.keras',           'Modèle 3 (Dropout + L2)'),
    ('history3_l2.pickle',        'Historique Modèle 3'),
    ('model4_augmented.keras',    'Modèle 4 (Augmentation + Profond)'),
    ('history4_augmented.pickle', 'Historique Modèle 4'),
    ('fig_dataset_examples.png',  'Exemples du dataset'),
    ('fig_class_distribution.png','Distribution des classes'),
    ('fig_model1_curves.png',     'Courbes Modèle 1'),
    ('fig_model2_curves.png',     'Courbes Modèle 2'),
    ('fig_model3_curves.png',     'Courbes Modèle 3'),
    ('fig_model4_curves.png',     'Courbes Modèle 4'),
    ('fig_comparison.png',        'Comparaison des 4 modèles'),
    ('fig_confusion_matrix.png',  'Matrice de confusion'),
    ('fig_predictions.png',       'Exemples de prédictions'),
    ('fig_per_class_accuracy.png','Précision par classe'),
]
for fname, desc in saved_files:
    print(f"  {fname:<35} → {desc}")